In [1]:
import os
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

print(tf.__version__)

2.16.1


In [2]:
IMG_SIZE = (224,224)
BATCH_SIZE = 16
SEED = 42

FINE_TUNE_EPOCHS = 15

dataset_path = "../../datasets/food-101/images"

print(os.path.exists(dataset_path))

True


In [3]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)

Found 101000 files belonging to 101 classes.
Using 80800 files for training.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.


In [4]:
model = load_model("../saved_models/efficientnetb0_initial.keras")

print("Model Loaded")

Model Loaded


In [5]:
for i, layer in enumerate(model.layers):
    print(i, layer.name)

0 input_layer_10
1 data_augmentation
2 efficientnetb0
3 global_average_pooling2d_3
4 dropout_3
5 dense_3


In [6]:
base_model = model.layers[2]

print(base_model.name)
print(len(base_model.layers))

efficientnetb0
238


In [7]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

print("Trainable Layers:",
      sum(layer.trainable for layer in base_model.layers))

Trainable Layers: 30


In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-Tune Model Compiled")

Fine-Tune Model Compiled


In [9]:
checkpoint_phase2 = ModelCheckpoint(
    "../saved_models/efficientnetb0_finetuned.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

csv_logger = CSVLogger(
    "../saved_models/training_log_finetune.csv"
)

print("Callbacks Ready")

Callbacks Ready


In [10]:
history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[
        checkpoint_phase2,
        early_stop,
        reduce_lr,
        csv_logger
    ],
    verbose=1
)

Epoch 1/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 729ms/step - accuracy: 0.4001 - loss: 2.5181
Epoch 1: val_accuracy improved from None to 0.65614, saving model to ../saved_models/efficientnetb0_finetuned.keras

Epoch 1: finished saving model to ../saved_models/efficientnetb0_finetuned.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 4753s 938ms/step - accuracy: 0.4540 - loss: 2.2247 - val_accuracy: 0.6561 - val_loss: 1.3402 - learning_rate: 1.0000e-05
Epoch 2/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 591ms/step - accuracy: 0.5334 - loss: 1.8333
Epoch 2: val_accuracy improved from 0.65614 to 0.66965, saving model to ../saved_models/efficientnetb0_finetuned.keras

Epoch 2: finished saving model to ../saved_models/efficientnetb0_finetuned.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 3423s 678ms/step - accuracy: 0.5452 - loss: 1.7789 - val_accuracy: 0.6697 - val_loss: 1.2736 - learning_rate: 1.0000e-05
Epoch 3/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.5626 - loss: 1.6864
Epoch 3: val_accuracy impro

In [1]:
import os

print(os.path.exists("../saved_models/efficientnetb0_finetuned.keras"))

True


In [2]:
from tensorflow.keras.models import load_model

model = load_model("../saved_models/efficientnetb0_finetuned.keras")

print("Model Loaded Successfully")

Model Loaded Successfully
